# Phase 2 — Supervisor + Router + Market Research

> **Gemini-only project.** Every cell here uses Google Gemini (chat + embeddings) via `langchain-google-genai`. The only key the core needs is `GOOGLE_API_KEY`. This notebook is a **scaffold** — run it top-to-bottom *after* Claude Code finishes this phase. If a cell references a module that doesn't exist yet, that phase hasn't been built.

**Goal:** route queries to the right agent and visualize the multi-agent graph.

**Note:** run the mixed query for `CLT-002` (it holds NVDA).


In [ ]:
# --- Setup: load .env and make the project importable ---
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # repo root
from dotenv import load_dotenv
load_dotenv('../.env', override=True)  # override=True: beat VS Code's stale cached env vars

# This project is Gemini-only. Confirm the one required key is present.
assert os.getenv('GOOGLE_API_KEY'), 'Set GOOGLE_API_KEY in .env (see 00_prerequisites_and_setup.md)'
print('GOOGLE_API_KEY loaded:', bool(os.getenv('GOOGLE_API_KEY')))
print('SEC_USER_AGENT   :', os.getenv('SEC_USER_AGENT', '(set before Phase 5)'))

## 1. Build the graph (Builder pattern)

In [ ]:
from app.graph.builder import GraphBuilder
graph = (GraphBuilder()
         .with_supervisor()
         .with_portfolio_agent()
         .with_market_research_agent()
         .build())

## 2. Routing decisions for three query types

In [ ]:
queries = [
    ('CLT-002', 'What stocks do I own?'),
    ('CLT-002', 'What is happening in the semiconductor sector today?'),
    ('CLT-002', 'How is my NVDA position doing and what is the news?'),
]
for cid, q in queries:
    out = graph.invoke({'messages': [('user', q)], 'client_id': cid,
                        'session_id': 'nb-2'})
    print(f'Q: {q}\n  route/agents -> {out.get("route")}\n')

## 3. Visualize the graph (Mermaid — always works; PNG if a renderer is available)

In [ ]:
mmd = graph.get_graph().draw_mermaid()
os.makedirs('../docs', exist_ok=True)
open('../docs/graph_phase2.mmd', 'w').write(mmd)
print(mmd)
try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print('(PNG render unavailable, Mermaid text above is the source):', e)

## ✅ Acceptance check

In [ ]:
# 'stocks I own' -> portfolio ; 'semiconductor sector' -> market_research
print('Inspect the routes printed above; both agents fire in sequence for the mixed query.')
print('Phase 2 acceptance: review routing + rendered graph.')